In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import (
    gini,
    apply_pf_schedule_to_mps,
    plot_stacked_gain_loss_sortable,
    print_baseline_energy_data,
    print_opt_energy_data,
    print_gini_on_smp_energy_flow,
)
from modules.visualisations import (
    plot_profile_by_category,
    plot_distribution_comparison,
    plot_sorted_mps_comparison,
)

from plotly.io import to_html
from IPython.display import display, HTML
import cvxpy as cp

from modules.optimization_algorithms.NashProductOptAlgo import NashProductOptAlgo
from modules.optimization_algorithms.XNashProductOptAlgo import XNashProductOptAlgo
from modules.optimization_algorithms.EqualWaterfillingOptAlgo import (
    EqualWaterfillingOptAlgo,
)

from datetime import datetime, UTC
from pathlib import Path
import json

In [ ]:
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

## Params

In [ ]:
# # PARMS
# changeable
org_id = 1
start_time = datetime(2025, 6, 21)
end_time = datetime(2025, 6, 22)

opt_algo = EqualWaterfillingOptAlgo() 

alphas=[0, 0.25, 0.5, 0.75, 1, 1.5]
opt_algo = XNashProductOptAlgo(alpha=alphas[2], solver=cp.SCS)

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

## Load data

In [ ]:
config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename = config_dict["SINGLE_MPS_OF_EEG"].format(org_id=org_id)

full_filepath_to_load = f"{path_to_local_data}{input_filename}"

In [ ]:
raw_eeg = pd.read_csv(f"{full_filepath_to_load}")
raw_eeg['time'] = pd.to_datetime(raw_eeg['time'], utc=True)

print(f"{raw_eeg.dtypes}")
print(f"len: {len(raw_eeg)}")
raw_eeg.head()

In [ ]:
eeg_selected_feat = raw_eeg[["time", "organization_id", "metering_point_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del raw_eeg
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.drop("metering_point_id", axis=1).describe())

work = eeg_selected_feat[
    (eeg_selected_feat["time"] > pd.Timestamp(start_time, tz='UTC')) &
    (eeg_selected_feat["time"] < pd.Timestamp(end_time, tz='UTC'))
]
del eeg_selected_feat
# cons_gen: Consumed Generation, how much of the generated electricity was consumed within the EEG
work["cons_gen"] = work["wt_meas_gen"] - work["wt_surp_gen"]
work.sum(numeric_only=True)

## eda

In [ ]:
feature = "cons_gen" # "wt_meas_gen" # "comm_cov"
energy_direction_filter = "G" # "C"

### Waterfilling Opt for finding optimal pfs

In [ ]:
mp_counts_on_time = (
    work
    .groupby("time")["energy_direction"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"C": "count_C_mps", "G": "count_G_mps"})
)
sums_on_time = work.groupby(by="time").sum().reset_index().drop(columns=["metering_point_id", "energy_direction"]).rename(columns={"wt_meas_cons":"sum_wt_meas_cons", "comm_pot":"sum_comm_pot", "comm_cov":"sum_comm_cov", "wt_meas_gen":"sum_wt_meas_gen", "wt_surp_gen":"sum_wt_surp_gen", "cons_gen":"sum_cons_gen"})

agg_on_time = pd.merge(left=sums_on_time, right=mp_counts_on_time, on="time", how="outer")

time_with_deficit = agg_on_time[agg_on_time["sum_wt_surp_gen"] <= 0]
time_with_surplus = agg_on_time[agg_on_time["sum_wt_surp_gen"] > 0]

print(f"{len(time_with_deficit)}/{len(agg_on_time)} ({(len(time_with_deficit)/len(agg_on_time))*100:.4}%) timestamps has deficit. only for consumers during deficit a pf is optimized")
print(f"{len(time_with_surplus)}/{len(agg_on_time)} ({(len(time_with_surplus)/len(agg_on_time))*100:.4}%) timestamps has surplus. only for generators during surplus a pf is optimized")


#### Acutal optimization calculation

In [ ]:
time_for_pf_opt = time_with_deficit if energy_direction_filter == "C" else time_with_surplus
feat_for_r0_pf_opt = "wt_meas_cons" if energy_direction_filter == "C" else "wt_meas_gen"
feat_for_R0_pf_opt = "sum_wt_meas_gen" if energy_direction_filter == "C" else "sum_wt_meas_cons"
feat_as_opt_target = "comm_cov" if energy_direction_filter == "C" else "cons_gen"
feat_for_aopt_pf_opt = f"{feat_as_opt_target}_opt"

In [ ]:
work_deficit = work #pd.merge(left=work[work["energy_direction"] == energy_direction_filter], right=time_for_pf_opt, on="time", how="inner")
tf_schedule = opt_algo.calculate_pfs(work_deficit)

### Apply pf schedule to energy data

In [ ]:
applied_pfs = apply_pf_schedule_to_mps(work, tf_schedule)

applied_pfs["comm_cov_delta"] = applied_pfs["opt_comm_cov"] - applied_pfs["comm_cov"]
applied_pfs["cons_gen_delta"] = applied_pfs["opt_cons_gen"] - applied_pfs["cons_gen"]

In [ ]:
print_baseline_energy_data(applied_pfs)
print_opt_energy_data(applied_pfs)
print_gini_on_smp_energy_flow(applied_pfs)

### Save optimized energy data to file


In [ ]:
cap = (
        lambda df: (
            df
            .groupby("time", as_index=False)["comm_cov"]
            .quantile(0.75)
            .rename(columns={"comm_cov": "q75_comm_cov"})
        )
    )

In [ ]:
cap(work)

In [ ]:
run_timestamp = datetime.now().strftime("%Y-%m-%dT%H:%M")
simulated_energy_data={}
simulated_energy_data[org_id] = {
    "tf_schedule": tf_schedule,
    "simulated_energy_data": applied_pfs,
    "algo_name": opt_algo.__name__,
    "algo_params": opt_algo.get_params(),  # assumes params stored on self
    "created_at": run_timestamp,
}

base_path = Path("../../local_data/")
base_path.mkdir(exist_ok=True)

for org_id, result in simulated_energy_data.items():

    org_path = base_path / f"org_{org_id}" / opt_algo.__name__
    org_path.mkdir(parents=True, exist_ok=True)

    # TF schedule
    result["tf_schedule"].to_csv(
        org_path / f"tf_schedule_{run_timestamp}.csv",
        index=False,
    )

    # 2Simulated energy data
    result["simulated_energy_data"].to_csv(
        org_path / f"simulated_energy_{run_timestamp}.csv",
        index=False,
    )

    # Metadata
    metadata = {
        "organization_id": org_id,
        "algo": result["algo_name"],
        "algo_params": result["algo_params"],
        "created_at": result["created_at"],
        "row_counts": {
            "tf_schedule": len(result["tf_schedule"]),
            "simulated_energy": len(result["simulated_energy_data"]),
        },
    }

    with open(org_path / f"metadata_{run_timestamp}.json", "w") as f:
        json.dump(metadata, f, indent=2)
